# R

A refresher on **R** — the language and environment built *for* statistics. R is a vectorized, interactive, functional language descended from **S** (Bell Labs, 1976), open-sourced in 1995, and now the lingua franca of academic statistics, biostatistics, econometrics, and exploratory data analysis. Its superpower is the **package ecosystem**: ~20,000 packages on CRAN plus Bioconductor for genomics, covering nearly every statistical method ever published.

**Domain:** Data Analysis & Research  ·  **from study list**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _R kernel / rpy2_

> This notebook is **not** Python-runnable. R needs its own interpreter (the `IRkernel` Jupyter kernel, an `.R` script via `Rscript`, or the REPL). Every code block below is **R or shell** meant to be run in those — there is no fake `print()` output here.

## 1. What & Why

**What it is.** R is a domain-specific language and runtime for **statistical computing and graphics**. It is an interactive, dynamically-typed, vectorized, functional language with first-class support for tabular data (`data.frame`), missing values (`NA`), statistical models (`lm`, `glm`, `formula` objects), and a deep, peer-reviewed package library on **CRAN**.

**The problem it solves.** Statisticians needed a notation where a *whole column* is the unit of computation, where "missing" is a built-in concept (not a sentinel hack), where fitting a model and plotting its diagnostics are one-liners, and where the entire body of statistical methods is installable. R is that environment — closer to a programmable graphing-statistics calculator than to a systems language.

**Reach for R when:**
- **Statistics is the point** — mixed-effects models (`lme4`), survival analysis (`survival`), Bayesian inference (`brms`, `rstan`), time series (`forecast`), GAMs (`mgcv`), and thousands of niche methods exist in R *first* (or only).
- **Exploratory data analysis & publication graphics** — `dplyr` for wrangling and `ggplot2` for plots that go straight into a paper.
- **Reproducible research / reporting** — R Markdown and **Quarto** weave code, prose, and figures into HTML/PDF/Word.
- **Bioinformatics** — Bioconductor is the standard toolchain for genomics.

**Don't reach for R when:**
- You're building **production services, ML pipelines for deployment, or general-purpose software** — Python's ecosystem (and its operational tooling) wins.
- You need **raw performance / low-level control** — R is interpreted and copy-on-modify; hot loops belong in C++ (`Rcpp`), or use a different language.
- The work is **engineering, not analysis** — web backends, systems code, big distributed pipelines.

## 2. Mental Model

**R is an interactive statistics console where everything is a vector and tables flow through functions.**

Three ideas make R click:

1. **Everything is a vector.** There are no scalars — `42` is a numeric vector of length 1. Operations are **vectorized**: `x + 1` adds 1 to every element, no loop. This is why idiomatic R has so few `for` loops.

2. **The `data.frame` is the table.** A data frame is a list of equal-length column vectors — a spreadsheet with named, typed columns. Nearly all analysis is "take a data frame, transform columns, model it, plot it." The tidyverse's `tibble` is a modernized data frame.

3. **Copy-on-modify + functional flavor.** Assigning or passing a value behaves as if you got a copy; functions don't mutate their arguments (mostly). You build pipelines that *return new* data rather than mutating in place.

The picture:

```
  raw data ──► data.frame/tibble ──► dplyr verbs (filter/mutate/summarise)
                                         │
                                         ├──► lm()/glm()  ──► model object ──► summary()/predict()
                                         └──► ggplot()     ──► layered graphic
```

Mental shorthand: **spreadsheet brain meets a functional language, with the entire statistics literature one `install.packages()` away.**

## 3. Key Concepts

- **Vectors & recycling.** The atom of R is the vector. When two vectors differ in length, the shorter is **recycled** (repeated) to match — powerful and a classic footgun (`c(1,2,3,4) + c(10,20)` → `11 22 13 24`).
- **Atomic types & coercion.** `logical < integer < double < character` — mixing types in `c()` silently coerces upward (`c(1, "a")` → all character).
- **`data.frame` / `tibble`.** A named list of equal-length columns. `tibble` (tidyverse) prints nicer and doesn't do surprise coercions.
- **Factors.** Categorical variables stored as integer codes + `levels`. Essential for modeling (they define contrasts), notorious for surprises (a factor that looks like numbers isn't).
- **`NA`, `NULL`, `NaN`.** `NA` = missing (typed, contagious: `1 + NA` is `NA`). `NULL` = absence/empty (length 0). `NaN` = not-a-number. They are distinct.
- **Indexing: `[`, `[[`, `$`.** `[` keeps the container (returns a sub-list/data.frame), `[[` extracts one element, `$` gets a named element by name. `df[ , "col"]` vs `df[["col"]]` vs `df$col`.
- **Vectorization & the `apply` family.** Prefer vectorized ops; when you must iterate, use `sapply`/`lapply`/`vapply`/`mapply` (base) or `purrr::map_*` (tidyverse) over hand-written loops.
- **Functions, lazy evaluation, `...`.** Arguments are evaluated lazily (only when used). `...` forwards extra args. Functions are first-class values.
- **Pipes.** `|>` is the **native** pipe (R ≥ 4.1): `x |> f() |> g()` means `g(f(x))`. `%>%` is the older **magrittr** pipe used across the tidyverse (supports `.` placeholder).
- **Lexical scoping & environments.** Functions capture the environment where they were *defined*. `<<-` reaches into enclosing scopes.
- **OOP systems.** **S3** (informal, dispatch on a `class` attribute — by far the most common), **S4** (formal, used by Bioconductor), **R5/Reference Classes** and **R6** (mutable, reference semantics).
- **Formulas.** `y ~ x1 + x2` is a first-class object describing a model; `*` adds interactions, `:` is interaction only, `.` means "all other columns".
- **Packages & namespaces.** `install.packages()` then `library(pkg)`; call without attaching via `pkg::fun()`. `pkg:::internal` reaches unexported functions.
- **base R vs tidyverse.** Two coherent dialects: terse base R (`[`, `apply`, `aggregate`) vs the readable, pipe-first **tidyverse** (`dplyr`, `tidyr`, `ggplot2`, `purrr`).

## 4. Setup

R is a separate interpreter — install it, not a `pip` package. (To drive R from Jupyter you install the `IRkernel`; to call R from Python you'd use `rpy2`.)

```bash
# --- Install R (the interpreter) ---
# macOS:
brew install --cask r              # or download from CRAN
# Debian/Ubuntu:
sudo apt-get install r-base r-base-dev
# Windows: installer from https://cran.r-project.org

# An IDE helps a lot:
brew install --cask rstudio        # RStudio / Posit, the standard R IDE
# (Positron is Posit's newer VS Code-based IDE)

# Verify:
R --version
Rscript -e 'R.version.string'
```

```r
# --- Inside R: install packages from CRAN ---
install.packages("tidyverse")      # dplyr, ggplot2, tidyr, readr, purrr, tibble, ...
install.packages(c("lme4", "data.table"))

# Use a package (attach it):
library(tidyverse)

# Bioconductor (genomics) uses its own installer:
install.packages("BiocManager")
BiocManager::install("DESeq2")
```

```r
# --- Reproducible, per-project dependencies with renv (the right way) ---
install.packages("renv")
renv::init()        # snapshots project library into renv.lock
renv::snapshot()    # record current package versions
renv::restore()     # rebuild the exact library on another machine
```

```bash
# --- Register the Jupyter kernel so .ipynb notebooks can run R ---
R -e 'install.packages("IRkernel"); IRkernel::installspec()'
# now "R" appears as a kernel choice in JupyterLab
```

## 5. Worked Examples

**These run in an R session, not this Python kernel.** Paste them into the R REPL, save as `script.R` and run `Rscript script.R`, or use an R Jupyter kernel. The expected results are described in prose — there's no fabricated cell output.

### Example 1 — Base R: vectors, a data frame, and the recycling rule

```r
# Everything is a vector; operations are vectorized (no loop needed).
x <- c(2, 4, 6, 8, 10)
mean(x)            # 6
x * 2              # 4 8 12 16 20  -- elementwise
x[x > 5]           # 6 8 10        -- logical indexing (1-BASED)

# Recycling: the shorter vector is repeated to match the longer one.
c(1, 2, 3, 4) + c(10, 20)   # 11 22 13 24   (10,20 recycled -> 10,20,10,20)

# A data.frame is a list of equal-length, typed columns.
df <- data.frame(
  name  = c("Ada", "Alan", "Grace"),
  score = c(91, 78, 88),
  pass  = c(TRUE, FALSE, TRUE)
)

df[df$pass, ]            # rows where pass is TRUE  (note the trailing comma)
df[ , "score"]           # the score column as a vector
df[["name"]]             # same as df$name
aggregate(score ~ pass, data = df, FUN = mean)   # mean score by pass/fail
```

Key takeaways: indexing is **1-based**, `df[rows, cols]` slices a frame, the trailing comma matters, and `~` already shows up in base aggregation. Recycling is silent — useful, but read the lengths.

### Example 2 — Tidyverse: a dplyr pipeline + ggplot2

The tidyverse rewrites the same work as a readable left-to-right pipeline using the pipe.

```r
library(dplyr)
library(ggplot2)

# `mtcars` ships with R. Wrangle, then summarise per group.
summary_tbl <- mtcars |>                      # native pipe (R >= 4.1)
  filter(mpg > 15) |>
  mutate(kml = mpg * 0.4251) |>               # add a derived column
  group_by(cyl) |>
  summarise(
    n        = n(),
    mean_hp  = mean(hp),
    mean_kml = mean(kml),
    .groups  = "drop"
  ) |>
  arrange(desc(mean_hp))

summary_tbl   # a tibble: one row per cylinder count

# Publication-quality graphics are layered: data + aesthetics + geoms.
ggplot(mtcars, aes(x = wt, y = mpg, colour = factor(cyl))) +
  geom_point(size = 2) +
  geom_smooth(method = "lm", se = FALSE) +
  labs(title = "Fuel economy vs weight", x = "Weight (1000 lb)",
       y = "Miles / gallon", colour = "Cylinders") +
  theme_minimal()
```

The mental model: `dplyr` verbs (`filter`, `mutate`, `group_by`, `summarise`, `arrange`) each take a data frame and return one, so they chain. `ggplot2` builds a plot by **adding layers** with `+`.

### Example 3 — Statistical modeling: this is what R is *for*

Fitting and interrogating a model is a few lines, and the `formula` object (`y ~ x`) is central.

```r
# Linear model: mpg explained by weight and cylinders.
fit <- lm(mpg ~ wt + factor(cyl), data = mtcars)

summary(fit)        # coefficients, std errors, t-values, p-values, R^2
confint(fit)        # 95% confidence intervals for each coefficient
coef(fit)           # just the point estimates

# Diagnostics and prediction:
par(mfrow = c(2, 2)); plot(fit)          # residuals, QQ, leverage -- one call
predict(fit, newdata = data.frame(wt = 3.0, cyl = 6))

# Same grammar generalizes:
glm(vs ~ wt + hp, data = mtcars, family = binomial)   # logistic regression
anova(fit)                                            # sequential ANOVA table
```

`factor(cyl)` tells R to treat cylinders as **categorical**, so it fits dummy contrasts rather than a slope. The `formula` interface (`mpg ~ wt + factor(cyl)`) is shared by `lm`, `glm`, `lme4::lmer`, `mgcv::gam`, and most modeling packages — learn it once.

### Example 4 — Scripting & reproducibility from the shell

R isn't only interactive. Drive it from the command line for batch jobs and pipelines.

```bash
# Run an expression directly:
Rscript -e 'cat(mean(1:100), "\n")'        # prints 50.5

# Run a whole script, passing arguments:
Rscript analysis.R input.csv results/
```

```r
# analysis.R -- a minimal reproducible batch script
args <- commandArgs(trailingOnly = TRUE)     # c("input.csv", "results/")
infile <- args[[1]]; outdir <- args[[2]]

library(readr); library(dplyr)

readr::read_csv(infile) |>
  filter(!is.na(value)) |>
  group_by(group) |>
  summarise(mean_value = mean(value), .groups = "drop") |>
  readr::write_csv(file.path(outdir, "summary.csv"))
```

For literate, reproducible reports, the same R code goes inside a **Quarto** (`.qmd`) or **R Markdown** (`.Rmd`) document and renders to HTML/PDF/Word:

```bash
quarto render report.qmd        # code + prose + figures -> one document
```

This is the backbone of reproducible research in R: a script or `.qmd` plus a pinned `renv.lock` reproduces the analysis exactly.

## 6. Gotchas & Pitfalls

- **1-based indexing.** `x[1]` is the first element; `x[0]` returns an empty vector, not an error. Coming from Python/C this bites constantly.
- **Silent recycling.** Mismatched lengths don't error — the short one repeats. Great until `df$a + some_shorter_vector` silently misaligns your data.
- **Factors masquerading as numbers.** `as.numeric(factor(c("10","20","30")))` gives `1 2 3` (the integer codes!), not `10 20 30`. Convert via `as.numeric(as.character(f))`.
- **`stringsAsFactors`.** Pre-R 4.0, `data.frame()` and `read.csv()` turned strings into factors by default — a legendary source of bugs. R 4.0 changed the default to `FALSE`; old code/tutorials may assume the old behavior.
- **`drop = TRUE` collapses frames.** `df[, "col"]` returns a *vector*, not a one-column data frame. Use `df[, "col", drop = FALSE]` (or tidyverse `select`) to keep it a frame.
- **`NA` is contagious and breaks comparisons.** `1 + NA` is `NA`; `NA == NA` is `NA` (not `TRUE`). Use `is.na(x)`, and pass `na.rm = TRUE` to `mean`/`sum`/etc. Never test `x == NA`.
- **Floating-point equality.** `0.1 + 0.2 == 0.3` is `FALSE`. Use `all.equal()` or `isTRUE(all.equal(...))`.
- **`T`/`F` are reassignable.** `TRUE`/`FALSE` are reserved, but `T` and `F` are ordinary variables someone can overwrite. Always write `TRUE`/`FALSE` in real code.
- **`<-` vs `=`.** Use `<-` for assignment; `=` is for naming function arguments. `x = 5` works at top level but conflates the two roles, and `f(x <- 5)` does something surprising.
- **`sapply` type instability.** `sapply` guesses its return type and may give you a list, vector, or matrix depending on the data — fine interactively, dangerous in scripts. Use `vapply` (declare the type) or `purrr::map_*`.
- **Namespace masking.** Loading `dplyr` masks `stats::filter` and `stats::lag`; loading `MASS` masks `dplyr::select`. Watch the startup messages and disambiguate with `dplyr::filter()`.
- **Partial matching of `$`.** `df$na` may match a column named `name` if it's the only match — a silent typo trap. `[[` does exact matching.
- **Hidden copies / performance.** Copy-on-modify means growing a vector in a loop (`x <- c(x, i)`) reallocates each time. Pre-allocate, vectorize, or reach for `data.table`/`Rcpp` for hot paths.

## 7. When to Use vs Alternatives

| Option | Best at | Weaknesses vs R |
|---|---|---|
| **R** | Statistics, EDA, publication graphics (`ggplot2`), reproducible reports (Quarto), niche stats methods, bioinformatics (Bioconductor) | Production engineering, deployment, general-purpose software, raw speed |
| **Python (pandas/polars/scikit-learn)** | General-purpose + glue, ML/deep learning, production pipelines, web/APIs, big ecosystem beyond stats | Fewer cutting-edge classical-stats packages; base plotting less elegant than `ggplot2` |
| **Julia** | High-performance numerical computing with a clean syntax; closes R/Python's speed gap | Much smaller package ecosystem and community |
| **SQL** | Set-based queries over large tables in a database; the right tool for joins/aggregations at scale | Not a modeling/plotting environment — pair it *with* R/Python |
| **SAS / SPSS / Stata** | Regulated industries (pharma, gov), GUI-driven analysts, long-term vendor support | Proprietary, expensive, far less flexible and extensible than R |

**Honest take.** Use **R when statistics or statistical graphics is the deliverable** — the modeling packages, the formula interface, `ggplot2`, and reproducible reporting are genuinely best-in-class. Use **Python when the analysis must live inside software** — ML in production, services, orchestration, or a team that's already Python. The two interoperate well (`reticulate` calls Python from R; `rpy2` calls R from Python), so "R *and* Python" is a common, healthy answer. Reach for **Julia** only when profiling proves you're speed-bound, and let **SQL** do the heavy table work upstream of either.

## 8. Resources

- **The R Project / CRAN** — the source of truth for the language and packages: https://www.r-project.org and https://cran.r-project.org
- **R for Data Science (2e)** — Wickham, Çetinkaya-Rundel & Grolemund; the canonical free tidyverse book: https://r4ds.hadley.nz
- **Advanced R (2e)** — Wickham; how the language *actually* works (vectors, environments, OOP, metaprogramming): https://adv-r.hadley.nz
- **Posit (RStudio) cheatsheets** — one-page references for dplyr, ggplot2, purrr, etc.: https://posit.co/resources/cheatsheets/
- **An Introduction to R** — the official manual (great for base-R fundamentals): https://cran.r-project.org/doc/manuals/r-release/R-intro.html
- **The R Inferno** — Burns's classic catalogue of R's traps and surprises: https://www.burns-stat.com/documents/books/the-r-inferno/
- **CRAN Task Views** — curated package lists by domain (Econometrics, Bayesian, Survival, …): https://cran.r-project.org/web/views/
- **Bioconductor** — the genomics/bioinformatics ecosystem: https://bioconductor.org

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE